In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import plotly.graph_objects as go

from nn import *  # BranchNet, TrunkNet, DeepONet

# Use CPU + float64 everywhere
device = torch.device("cpu")
torch.set_default_dtype(torch.float64)
dtype = torch.float64


In [ ]:
def pde_loss(model, branch_in, trunk_in):
    # branch_in = [s, t], trunk_in = [r, sigma]
    branch_in = branch_in.requires_grad_(True)

    r, sigma = trunk_in[:, 0:1], trunk_in[:, 1:2]
    s, t = branch_in[:, 0:1], branch_in[:, 1:2]
    V = model(branch_in, trunk_in)

    grad = torch.autograd.grad(V.sum(), branch_in, create_graph=True)[0]
    dvdt = grad[:, 1:2]
    dVds = grad[:, 0:1]
    d2vds2 = torch.autograd.grad(dVds.sum(), branch_in, create_graph=True)[0][:, 0:1]

    # Black–Scholes operator (European option)
    pde_res = dvdt + r*s*dVds + 0.5 * sigma**2 * s**2 * d2vds2 - r * V
    return torch.mean(pde_res**2)


In [ ]:
# %%
# Problem / sampling parameters
S_MIN = 0.0
K = 2.5
S_MAX = K * 4.0
T = 1.0

# Uniform ranges for rates and vols
R_MIN, R_MAX = 0.0, 0.20
SIGMA_MIN, SIGMA_MAX = 0.0, 0.50

# Dataset sizes
N_PDE   = 10_000
N_IC    = 10_000
N_BC_0  = 10_000
N_BC_SM = 10_000

# Batch size (used only if you switch to mini-batch training)
BATCH_SIZE = 100

rng = np.random.default_rng(seed=1234)


In [26]:
branch_dim = 2
trunk_dim = 2
latent_dim = 100
hidden_dim = 100

branch_net = BranchNet(in_dim=branch_dim, hidden_dim=hidden_dim,
                       out_dim=latent_dim).to(device)
trunk_net = TrunkNet(in_dim=trunk_dim, hidden_dim=hidden_dim,
                     out_dim=latent_dim).to(device)
model = DeepONet(branch_net, trunk_net).to(device)

mse_loss = torch.nn.MSELoss()

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)

model.apply(init_weights)
model.to(device)

DeepONet(
  (branch): BranchNet(
    (net): Sequential(
      (0): Linear(in_features=2, out_features=100, bias=True)
      (1): Tanh()
      (2): Linear(in_features=100, out_features=100, bias=True)
      (3): Tanh()
      (4): Linear(in_features=100, out_features=100, bias=True)
    )
  )
  (trunk): TrunkNet(
    (net): Sequential(
      (0): Linear(in_features=2, out_features=100, bias=True)
      (1): Tanh()
      (2): Linear(in_features=100, out_features=100, bias=True)
      (3): Tanh()
      (4): Linear(in_features=100, out_features=100, bias=True)
    )
  )
)

In [ ]:
# %%
# Model (DeepONet) in float64
branch_dim = 2   # [s, t]
trunk_dim  = 2   # [r, sigma]
latent_dim = 100
hidden_dim = 100

branch_net = BranchNet(in_dim=branch_dim, hidden_dim=hidden_dim, out_dim=latent_dim).to(device).to(dtype)
trunk_net  = TrunkNet(in_dim=trunk_dim,  hidden_dim=hidden_dim,  out_dim=latent_dim).to(device).to(dtype)
model = DeepONet(branch_net, trunk_net).to(device).to(dtype)

mse_loss = nn.MSELoss()

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)

model.apply(init_weights)
for p in model.parameters():
    p.data = p.data.to(dtype)


In [ ]:
def train_lbfgs(model, n_iters=10_000):
    optimizer = optim.LBFGS(
        model.parameters(), lr=1.0,
        max_eval=n_iters, max_iter=n_iters,
        line_search_fn='strong_wolfe'
    )
    model.train()

    def closure():
        optimizer.zero_grad()
        loss_val = pde_loss(model, pde_inputs, trunk_pde_input)
        loss_val += mse_loss(model(boundary_inputs, trunk_boundary_input), boundary_outputs)
        loss_val.backward()
        print('Loss:', float(loss_val.detach().cpu().numpy()), end='\r', flush=True)
        return loss_val

    optimizer.step(closure)
    print("\nTraining complete.")

train_lbfgs(model, n_iters=10_000)


Training complete.23698e-057.02108805812895298


In [ ]:
# %%
# Plot price surface for fixed r, sigma
s = torch.linspace(S_MIN, S_MAX, 100, dtype=dtype, device=device)
t = torch.linspace(0.0, T, 100, dtype=dtype, device=device)

ss, tt = torch.meshgrid(s, t, indexing="ij")
branch_input = torch.concat([ss.reshape(-1, 1), tt.reshape(-1, 1)], dim=1)

r_const = 0.02
sig_const = 0.20
trunk_input = torch.concat([
    torch.full_like(ss.reshape(-1, 1), r_const),
    torch.full_like(ss.reshape(-1, 1), sig_const)
], dim=1)

with torch.no_grad():
    output = model(branch_input, trunk_input).cpu().numpy()

z = output.reshape(100, 100)
fig = go.Figure(data=[go.Surface(z=z, x=t.cpu().numpy(), y=s.cpu().numpy())])
fig.update_layout(
    title='DeepONet Option Pricing (constant r, σ)',
    autosize=False,
    scene=dict(xaxis_title='Time', yaxis_title='Stock', zaxis_title='Option Price'),
    width=800, height=800
)
fig.show()


## Time-dependent volatility

In [30]:
def vol_smile(k):
    k_min = 80
    k_max = 120
    sigma_min = 0.10
    sigma_max = 0.50    
    return sigma_max - (sigma_max - sigma_min) * np.sin(np.pi * (k - k_min) / (k_max - k_min))**2

def yield_curve(t, y_min=0.02, y_max=0.06, c=0.2):   
    return y_min + (y_max - y_min) * (1 - np.exp(-c * t))
# plot
t = np.linspace(0, 1, 100)
v = vol_smile(t)
r = yield_curve(t)

fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=v, mode='lines', name='Volatility'))
fig.update_layout(title='Volatility and Interest Rate over Time',
                  xaxis_title='Time',
                  yaxis_title='Value')
fig.show()

# plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=r, mode='lines', name='Interest Rate'))
fig.update_layout(title='Interest Rate over Time',
                  xaxis_title='Time',
                  yaxis_title='Value')
fig.show()

In [31]:
s = torch.linspace(0, K*4, 100)
t = torch.linspace(0, 1, 100)

ss, tt = torch.meshgrid(s, t)
ss = ss.reshape(-1).to(device)
tt = tt.reshape(-1).to(device)

vv = vol_smile(tt.cpu().numpy())
rr = yield_curve(tt.cpu().numpy())


branch_input = torch.concat([ss.reshape(-1, 1), tt.reshape(-1, 1)], dim=1)
trunk_input = torch.concat([torch.tensor(rr).reshape(-1, 1), torch.tensor(vv).reshape(-1, 1)], dim=1)
branch_input = branch_input.to(device)
trunk_input = trunk_input.to(device)
output = model(branch_input, trunk_input).cpu().detach().numpy()
z = output.reshape(100, 100)
fig = go.Figure(data=[go.Surface(z=z, x=t.cpu().numpy(), y=s.cpu().numpy())])
fig.update_layout(title='DeepONet Option Pricing with Volatility Smile', autosize=False,
                  scene=dict(
                        xaxis_title='Time',
                        yaxis_title='Stock',
                        zaxis_title='Option Price'),
                  width=800, height=800)
fig.show()